# TatVTON — Full Realistic Pipeline (Colab GPU)

Runs the **composite-then-refine** pipeline on a free Colab **T4 GPU**:

1. **SAM 2** finds the skin where you point.
2. The exact design is placed on the skin (the CPU preview).
3. **SDXL inpainting** repaints only the tattoo area at low strength, guided by **IP-Adapter**, so the ink looks settled into real skin while the design is kept.

| Model | Hub ID | Size (fp16) |
|---|---|---|
| SAM 2 large | `facebook/sam2-hiera-large` | ~0.9 GB |
| SDXL inpainting | `diffusers/stable-diffusion-xl-1.0-inpainting-0.1` | ~7 GB |
| SDXL VAE (fp16 fix) | `madebyollin/sdxl-vae-fp16-fix` | ~0.3 GB |
| IP-Adapter Plus (ViT-H) + image encoder | `h94/IP-Adapter` | ~2.5 GB |

All public, no Hugging Face login needed. First run downloads ~11 GB (a few minutes on Colab).

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
#@title 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}  VRAM: {props.total_memory / 1024**3:.1f} GB")

In [ ]:
#@title 2. Install TatVTON + SAM 2 (~2 min)
#@markdown If the GitHub repo is private, upload a zip of the repo instead and set `source` to `zip`.
source = "github"  #@param ["github", "zip"]
repo = "https://github.com/aman7935/Tatvton.git"  #@param {type:"string"}

import os
os.environ["SAM2_BUILD_CUDA"] = "0"  # skip the slow optional CUDA extension
if source == "github":
    !pip install -q "tatvton @ git+{repo}"
else:
    from google.colab import files
    up = files.upload()
    zip_name = next(iter(up))
    !unzip -q -o "{zip_name}" -d /content/tatvton_src
    !pip install -q /content/tatvton_src/*/
!pip install -q "sam-2 @ git+https://github.com/facebookresearch/sam2.git"

import tatvton
print("tatvton", tatvton.__version__)

In [ ]:
#@title 3. Upload a body photo, then a tattoo design
from google.colab import files
from PIL import Image

print("Upload the BODY photo:")
body_name = next(iter(files.upload()))
print("Upload the TATTOO design (plain or transparent background):")
tattoo_name = next(iter(files.upload()))

body = Image.open(body_name)
tattoo = Image.open(tattoo_name)
print("body:", body.size, body.mode, "| tattoo:", tattoo.size, tattoo.mode)

In [ ]:
#@title 4. Pick the tattoo position (read X/Y off the grid)
import matplotlib.pyplot as plt

X = 0  #@param {type:"integer"}
Y = 0  #@param {type:"integer"}
if X == 0 and Y == 0:  # default: image centre
    X, Y = body.width // 2, body.height // 2

plt.figure(figsize=(10, 8))
plt.imshow(body)
plt.scatter([X], [Y], s=300, c="red", marker="+", linewidths=3)
plt.grid(True, alpha=0.4)
plt.title(f"Red cross = tattoo position ({X}, {Y}). Edit X/Y above and re-run if needed.")
plt.show()

In [ ]:
#@title 5. Run the full realistic pipeline
import torch
from tatvton import TatVTONPipeline, TatVTONConfig, PointPrompt

strength = 0.4  #@param {type:"slider", min:0.15, max:0.8, step:0.05}
scale = 0.8  #@param {type:"slider", min:0.2, max:2.0, step:0.05}
rotation = 0  #@param {type:"slider", min:-180, max:180, step:1}
steps = 30  #@param {type:"integer"}
seed = 42  #@param {type:"integer"}

pipe = TatVTONPipeline(TatVTONConfig(num_inference_steps=steps))
region = PointPrompt(coords=[(X, Y)])
skin_mask = pipe.extract_mask(body, region)  # reused by the next cell

result = pipe(body, tattoo, region, mask=skin_mask, strength=strength,
              scale=scale, rotation=rotation, seed=seed)

fig, ax = plt.subplots(1, 2, figsize=(18, 8))
ax[0].imshow(result.metadata["preview"]); ax[0].set_title("Preview (placed design)")
ax[1].imshow(result.image); ax[1].set_title(f"Realistic (SDXL refine, strength={strength})")
for a in ax: a.axis("off")
plt.show()

result.image.save("tatvton_result.png")
print("Saved tatvton_result.png | seed:", result.seed)

In [ ]:
#@title 6. (Optional) Compare refinement strengths
#@markdown Lower = closer to your exact design; higher = more realistic skin but the AI changes more.
strengths = [0.25, 0.4, 0.55]
outs = [pipe(body, tattoo, region, mask=skin_mask, strength=s, scale=scale,
             rotation=rotation, seed=seed).image for s in strengths]
fig, ax = plt.subplots(1, len(outs), figsize=(8 * len(outs), 7))
for a, im, s in zip(ax, outs, strengths):
    a.imshow(im); a.set_title(f"strength={s}"); a.axis("off")
plt.show()
for im, s in zip(outs, strengths):
    im.save(f"tatvton_strength_{s}.png")

In [ ]:
#@title 7. Download results
from google.colab import files
import glob
for f in sorted(glob.glob("tatvton_*.png")):
    files.download(f)